In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.shape

(891, 12)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
df.isnull().sum()


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
df.duplicated().sum()

np.int64(0)

## Data Quality Report

- Total rows: 891, Total columns: 12
- Missing values:
  - Age: 177 missing (~20%)
  - Cabin: 687 missing (~77%)
  - Embarked: 2 missing (~0.2%)
- Duplicate rows: 0

In [7]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [8]:
df['Embarked'].isnull().sum()

np.int64(0)

In [9]:
df['Age'] = df['Age'].fillna(df['Age'].median())

In [10]:
df['Age'].isnull().sum()

np.int64(0)

## Handling Missing Values

- **Embarked** (2 missing): filled with mode, since it's categorical and only 2 rows were affected.
- **Age** (177 missing): filled with median, since age can have outliers (very young/old passengers) that would skew a mean-based fill.
- **Cabin** (687 missing, ~77%): dropped entirely, since too much of the column was missing to reliably impute. Considered creating a "Has_Cabin" flag instead, but kept the approach simple for this task.

In [11]:
df = df.drop('Cabin', axis=1)

In [12]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='object')

In [13]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [14]:
df['Sex'].unique()
df['Embarked'].unique()

array(['S', 'C', 'Q'], dtype=object)

In [15]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


## Standardization & Data Types

- Checked Sex and Embarked for inconsistent formatting (e.g. mixed casing) — none found, values were already clean.
- Checked all column data types using df.dtypes — all columns already had appropriate types (numeric columns as int64/float64, text columns as object). No conversion was needed.

In [16]:
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_age = df[(df['Age'] < lower_bound) | (df['Age'] > upper_bound)]
print(f"Age outlier bounds: {lower_bound} to {upper_bound}")
print(f"Number of Age outliers: {len(outliers_age)}")

Age outlier bounds: 2.5 to 54.5
Number of Age outliers: 66


In [17]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_fare = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print(f"Fare outlier bounds: {lower_bound} to {upper_bound}")
print(f"Number of Fare outliers: {len(outliers_fare)}")

Fare outlier bounds: -26.724 to 65.6344
Number of Fare outliers: 116


## Outlier Detection

Used the IQR method on Age and Fare:
- **Age**: 66 outliers found (bounds: 2.5–54.5). These represent older passengers — a legitimate age range, not a data error. Decision: kept as-is.
- **Fare**: 116 outliers found (bounds: -26.72–65.63). High fares reflect real first-class ticket prices. Removing them would lose meaningful information about passenger wealth. Decision: kept as-is, noted as a natural right-skew in the data.

In [18]:
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Total Columns', 'Missing Values (Age)', 'Missing Values (Cabin)', 'Missing Values (Embarked)', 'Duplicate Rows'],
    'Before Cleaning': [891, 12, 177, 687, 2, 0],
    'After Cleaning': [df.shape[0], df.shape[1], df['Age'].isnull().sum(), 'Column Dropped', df['Embarked'].isnull().sum(), df.duplicated().sum()]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Total Rows,891,891
1,Total Columns,12,11
2,Missing Values (Age),177,0
3,Missing Values (Cabin),687,Column Dropped
4,Missing Values (Embarked),2,0
5,Duplicate Rows,0,0


In [19]:
df.to_csv('cleaned_titanic.csv', index=False)